In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Once your Drive is mounted, you can access your files. Here's an example of how to load a CSV file named `my_data.csv` located in the root of your Google Drive into a pandas DataFrame. You might need to adjust the path if your file is in a different folder.

In [2]:
import pandas as pd
import os

# List files in your Google Drive (optional, helps to find the correct path)
# You can navigate through directories like this:
# os.listdir('/content/drive/MyDrive/')

# Example: Load a CSV file from the root of your Google Drive
file_path = '/content/drive/MyDrive/Indian_Rural_Urban_Combined_Dataset.csv'
# file_path_pre='/content/drive/MyDrive/updated_carbon_dataset.csv'

try:
    df = pd.read_csv(file_path)
    # df2=pd.read_csv(file_path_pre)
    print(f"Successfully loaded data from {file_path}:")
    display(df.head())
    # display(df2.head())
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please check the file path and try again.")
except Exception as e:
    print(f"An error occurred while loading the file: {e}")

Successfully loaded data from /content/drive/MyDrive/Indian_Rural_Urban_Combined_Dataset.csv:


,body_type,gender,diet_type,shower_frequency,heating_source,transport_mode,vehicle_fuel_type,social_activity_level,monthly_grocery_bill,air_travel_frequency,vehicle_distance_km,waste_bag_size,waste_bag_count,tv_pc_hours_daily,new_clothes_monthly,internet_hours_daily,energy_efficiency_level,recycling_habits,cooking_methods,total_carbon_emission
0,overweight,female,pescatarian,daily,coal,public,petrol,often,230,frequently,210,large,4,7,26,1,No,['Metal'],"['Stove', 'Oven']",864.77
1,obese,female,vegetarian,less frequently,natural gas,walk/bicycle,diesel,often,114,rarely,9,extra large,3,9,38,5,No,['Metal'],"['Stove', 'Microwave']",751.41
2,overweight,male,omnivore,more frequently,wood,private,petrol,never,138,never,2472,small,1,14,47,6,Sometimes,['Metal'],"['Oven', 'Microwave']",1245.65
3,overweight,male,omnivore,twice a day,wood,walk/bicycle,electric,sometimes,157,rarely,74,medium,3,20,5,7,Sometimes,"['Paper', 'Plastic', 'Glass', 'Metal']","['Microwave', 'Grill', 'Airfryer']",481.29
4,obese,female,vegetarian,daily,coal,private,diesel,often,266,very frequently,8457,large,1,3,5,6,Yes,['Paper'],['Oven'],2577.45


In [3]:
import numpy as np

In [4]:
# print("Number of null values per column:")
# print(df.isnull().sum())

# print("\nRows with any null values:")
# display(df[df.isnull().any(axis=1)])

print("Shape of df before filtering vehicle_fuel_type:", df.shape)
df['vehicle_fuel_type'].unique()
nanindex=df[df['vehicle_fuel_type'].isnull()].index
to_distribute=['petrol','diesel','electric','cng']
nan_len=len(nanindex)
to_distribute_len=len(to_distribute)
assign=np.array([to_distribute[i%to_distribute_len] for i in range(nan_len)])
df.loc[nanindex,'vehicle_fuel_type']=assign
df['vehicle_fuel_type'].value_counts()
df=df[~((df['vehicle_fuel_type']=='hybrid') | (df['vehicle_fuel_type']=='lpg'))]
print("Shape of df after filtering vehicle_fuel_type:", df.shape)
print("Vehicle fuel types after filtering:")
display(df['vehicle_fuel_type'].value_counts())

Shape of df before filtering vehicle_fuel_type: (13661, 20)
Shape of df after filtering vehicle_fuel_type: (12921, 20)
Vehicle fuel types after filtering:


,count
vehicle_fuel_type,
petrol,4403
diesel,4241
electric,2597
cng,1680


In [17]:
#model Preperation
import pandas as pd
import numpy as np
import ast
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [6]:
def get_list_count(val):
  try:
    items=ast.literal_eval(val) if isinstance(val,str) else val
    return len(items) if isinstance(items,list) else 0
  except:
    return 0

In [7]:
df['recycling_count']=df['recycling_habits'].apply(get_list_count)
df['cooking_count']=df['cooking_methods'].apply(get_list_count)

In [8]:
# Define Features and Target

categorical_features=[
    'body_type','gender','diet_type',
    'transport_mode','vehicle_fuel_type','social_activity_level','air_travel_frequency',
    'waste_bag_size','energy_efficiency_level'
]
numerical_features=['monthly_grocery_bill','vehicle_distance_km','waste_bag_count',
                    'tv_pc_hours_daily','new_clothes_monthly','internet_hours_daily',
                    'recycling_count','cooking_count'
                    ]
X=df[categorical_features+numerical_features]
y=df['total_carbon_emission']

In [9]:
for column in X.columns:
    print(f"Unique values for '{column}':")
    display(X[column].unique())
    print("\n")

Unique values for 'body_type':


array(['overweight', 'obese', 'underweight', 'normal'], dtype=object)



Unique values for 'gender':


array(['female', 'male'], dtype=object)



Unique values for 'diet_type':


...

array([ 1,  5,  6,  7,  9, 15, 18, 21,  4,  8, 14, 22,  3, 20, 23, 10, 17,
       16, 11, 24, 12,  0,  2, 19, 13])



Unique values for 'recycling_count':


array([1, 4, 3, 2, 0])



Unique values for 'cooking_count':


array([2, 3, 1, 4, 5, 0])

In [10]:
output_excel_path = '/content/drive/MyDrive/X_data.xlsx'

try:
    X.to_excel(output_excel_path, index=False)
    print(f"Successfully updated DataFrame X in {output_excel_path}")
except Exception as e:
    print(f"An error occurred while updating the Excel file: {e}")

Successfully updated DataFrame X in /content/drive/MyDrive/X_data.xlsx


In [18]:
# Preprocessing Pipeline
preprocessor=ColumnTransformer(
    transformers=[('num',StandardScaler(),numerical_features),
                  ('cat',OneHotEncoder(handle_unknown='ignore'),categorical_features)]
)
mlp_brain=MLPRegressor(hidden_layer_sizes=(128,64,32),
                       activation='relu',
                       solver='adam',
                       max_iter=1000,
                       random_state=42)
lR=LinearRegression()
lrmodel=Pipeline(steps=[('preprocessor',preprocessor),('lr',lR)])
model_pipeline=Pipeline(steps=[('preprocessor',preprocessor),('mlp',mlp_brain)])


In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
lrmodel.fit(X_train, y_train)
y_pred = lrmodel.predict(X_test)
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f} kg")

R2 Score: 0.9271
MAE: 86.66 kg


In [12]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Training (The model learns the weights here)
model_pipeline.fit(X_train, y_train)

# Testing
y_pred = model_pipeline.predict(X_test)
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f} kg")

R2 Score: 0.9976
MAE: 20.23 kg


In [13]:
# This file is what you will upload to your web server
joblib.dump(model_pipeline, 'indian_carbon_mp_model.pkl')

['indian_carbon_mp_model.pkl']

In [14]:
import sklearn
print(sklearn.__version__)

1.6.1
